In [0]:
dbutils.widgets.text("p_data_souce", "")

In [0]:
v_data_source = dbutils.widgets.get("p_data_souce")

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-21")

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configs"

In [0]:
%run "../SetUp/setup"

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


[FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/__unitystorage/', name='__unitystorage/', size=0, modificationTime=1769227742000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta/', name='drivers_convert_to_delta/', size=0, modificationTime=1769359802000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta_new/', name='drivers_convert_to_delta_new/', size=0, modificationTime=1769360049000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_external/', name='results_external/', size=0, modificationTime=1769228062000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_partitioned/', name='results_partitioned/', size=0, modificationTime=1769228706000)]

**Read the JSON file using spark dataframe reader**

In [0]:
#DDL style defining the schema:
#constructor_schema = "constructorId INT, constructorRef STRING, name STRING, nationality STRING, url STRING"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
constructor_schema = StructType([
    StructField("constructorId", IntegerType(), False),
    StructField("constructorRef", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True)
])

In [0]:
constructor_df = spark.read.schema(constructor_schema).json(f"{raw_folder_path}/{v_file_date}/constructors.json")

In [0]:
display(constructor_df)

constructorId,constructorRef,name,nationality,url
1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren
2,bmw_sauber,BMW Sauber,German,http://en.wikipedia.org/wiki/BMW_Sauber
3,williams,Williams,British,http://en.wikipedia.org/wiki/Williams_Grand_Prix_Engineering
4,renault,Renault,French,http://en.wikipedia.org/wiki/Renault_in_Formula_One
5,toro_rosso,Toro Rosso,Italian,http://en.wikipedia.org/wiki/Scuderia_Toro_Rosso
6,ferrari,Ferrari,Italian,http://en.wikipedia.org/wiki/Scuderia_Ferrari
7,toyota,Toyota,Japanese,http://en.wikipedia.org/wiki/Toyota_Racing
8,super_aguri,Super Aguri,Japanese,http://en.wikipedia.org/wiki/Super_Aguri_F1
9,red_bull,Red Bull,Austrian,http://en.wikipedia.org/wiki/Red_Bull_Racing
10,force_india,Force India,Indian,http://en.wikipedia.org/wiki/Racing_Point_Force_India


**Drop the unwanted columns from the dataframe**

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
constructor_dropped_df = constructor_df.drop(col("url"))

In [0]:
display(constructor_dropped_df)

constructorId,constructorRef,name,nationality
1,mclaren,McLaren,British
2,bmw_sauber,BMW Sauber,German
3,williams,Williams,British
4,renault,Renault,French
5,toro_rosso,Toro Rosso,Italian
6,ferrari,Ferrari,Italian
7,toyota,Toyota,Japanese
8,super_aguri,Super Aguri,Japanese
9,red_bull,Red Bull,Austrian
10,force_india,Force India,Indian


In [0]:
constructor_final_df = constructor_dropped_df.withColumnRenamed("constructorId", "constructor_id").withColumnRenamed("constructorRef", "constructor_ref").withColumn("ingestion_date", current_timestamp()).withColumn("data_source", lit(v_data_source)).withColumn("file_date", lit(v_file_date))

In [0]:
display(constructor_final_df)

constructor_id,constructor_ref,name,nationality,ingestion_date,data_source,file_date
1,mclaren,McLaren,British,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
2,bmw_sauber,BMW Sauber,German,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
3,williams,Williams,British,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
4,renault,Renault,French,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
5,toro_rosso,Toro Rosso,Italian,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
6,ferrari,Ferrari,Italian,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
7,toyota,Toyota,Japanese,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
8,super_aguri,Super Aguri,Japanese,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
9,red_bull,Red Bull,Austrian,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21
10,force_india,Force India,Indian,2026-01-26T05:57:45.310945Z,Ergast API,2021-03-21


**Write output in delta format**

In [0]:
constructors_final_df = constructor_final_df.dropDuplicates(["constructor_id"])

In [0]:
constructors_final_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("f1_processed.constructors")